# 方案 A — 路由策略验证实验

三组实验共用同一测试集（40 题），只替换 Prompt 中的工具描述。
- G1: Controller 路由（基线）
- G2: MCP Tool 路由（核心）
- G3: DSL 路由（备选）

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    base_url="https://gptapi.tutu02.us.ci/v1",
    api_key="Tianhe.00"
)

# 验证连接
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "回复 ok"}],
    max_tokens=10
)
print("连接成功:", resp.choices[0].message.content)

## 1. 测试集定义

In [ ]:
# 测试集：40 题
test_cases = [
    # === L1 单操作直查 ===
    {"id": "T01", "level": "L1", "question": "现在系统里一共有多少台设备？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/summary", "params": {}}],
         "G2": [{"tool": "get_dashboard_summary", "arguments": {}}],
         "G3": "QUERY equipment AGGREGATE COUNT"
     }},
    {"id": "T02", "level": "L1", "question": "当前有几个待审核的故障报修？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/summary", "params": {}}],
         "G2": [{"tool": "get_dashboard_summary", "arguments": {}}],
         "G3": "QUERY fault_report WHERE status = 0 AGGREGATE COUNT"
     }},
    {"id": "T03", "level": "L1", "question": "设备 EQ-001 的详细信息是什么？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/equipment/get", "params": {"id": "EQ-001"}}],
         "G2": [{"tool": "get_equipment_detail", "arguments": {"equipmentId": "EQ-001"}}],
         "G3": "QUERY equipment WHERE equipment_code = 'EQ-001'"
     }},
    {"id": "T04", "level": "L1", "question": "各状态的设备数量分布是怎样的？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/equipment-status-distribution", "params": {}}],
         "G2": [{"tool": "get_equipment_status_distribution", "arguments": {}}],
         "G3": "QUERY equipment AGGREGATE COUNT ORDER BY status"
     }},
    {"id": "T05", "level": "L1", "question": "最近 30 天的故障趋势怎么样？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/fault-trend", "params": {}}],
         "G2": [{"tool": "get_fault_trend", "arguments": {"days": 30}}],
         "G3": "QUERY fault_report WHERE report_time >= '2026-02-23' AGGREGATE COUNT ORDER BY report_time"
     }},
    {"id": "T06", "level": "L1", "question": "巡检异常的整体统计指标是什么？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/anomaly/statistics", "params": {}}],
         "G2": [{"tool": "get_anomaly_statistics", "arguments": {}}],
         "G3": "QUERY anomaly_record AGGREGATE COUNT"
     }},
    {"id": "T07", "level": "L1", "question": "当前有哪些库存预警？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/spare/alert/list", "params": {}}],
         "G2": [{"tool": "get_spare_alerts", "arguments": {}}],
         "G3": "QUERY spare_stock WHERE quantity < safety_line"
     }},
    {"id": "T08", "level": "L1", "question": "我有哪些待办事项？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/todo-list", "params": {}}],
         "G2": [{"tool": "get_todo_list", "arguments": {}}],
         "G3": "UNSUPPORTED"
     }},
    {"id": "T09", "level": "L1", "question": "维修工单目前各状态有多少个？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/repair-order/status-stats", "params": {}}],
         "G2": [{"tool": "query_repair_orders", "arguments": {}}],
         "G3": "QUERY repair_order AGGREGATE COUNT ORDER BY status"
     }},
    {"id": "T10", "level": "L1", "question": "设备分类中哪类设备最多？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/dashboard/category-distribution", "params": {}}],
         "G2": [{"tool": "get_dashboard_summary", "arguments": {}}],
         "G3": "QUERY equipment AGGREGATE COUNT ORDER BY category_id DESC LIMIT 1"
     }},

    # === L2 双操作关联 ===
    {"id": "T11", "level": "L2", "question": "上月故障最多的设备是哪台？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/fault-report/page", "params": {"reportTimeBegin": "2026-02-01", "reportTimeEnd": "2026-02-28"}}],
         "G2": [{"tool": "query_fault_reports", "arguments": {"dateRange": {"start": "2026-02-01", "end": "2026-02-28"}}}],
         "G3": "QUERY fault_report WHERE report_time BETWEEN '2026-02-01' AND '2026-02-28' AGGREGATE COUNT ORDER BY COUNT DESC LIMIT 1"
     }},
    {"id": "T12", "level": "L2", "question": "维修工单 WO-001 用了哪些备件？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/repair-order/spare/list", "params": {"repairOrderId": "WO-001"}}],
         "G2": [{"tool": "get_repair_detail", "arguments": {"repairOrderId": "WO-001"}}],
         "G3": "QUERY repair_spare_usage WHERE repair_order_id = 'WO-001' JOIN spare_part"
     }},
    {"id": "T13", "level": "L2", "question": "设备 EQ-002 的保养任务执行情况怎样？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/maintenance/task/page", "params": {"equipmentId": "EQ-002"}}],
         "G2": [{"tool": "query_maintenance_tasks", "arguments": {"equipmentId": "EQ-002"}}],
         "G3": "QUERY maintenance_task WHERE equipment_id = 'EQ-002'"
     }},
    {"id": "T14", "level": "L2", "question": "近 7 天完成了几个巡检任务？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/patrol/task/page", "params": {"status": "completed", "startTimeBegin": "2026-03-18"}}],
         "G2": [{"tool": "query_patrol_tasks", "arguments": {"status": 2, "dateRange": {"start": "2026-03-18", "end": "2026-03-25"}}}],
         "G3": "QUERY patrol_task WHERE status = 'completed' AND start_time >= '2026-03-18' AGGREGATE COUNT"
     }},
    {"id": "T15", "level": "L2", "question": "当前维修中的设备都是哪些？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/equipment/page", "params": {"status": 2}}],
         "G2": [{"tool": "query_equipment", "arguments": {"status": 2}}],
         "G3": "QUERY equipment WHERE status = 2"
     }},
    {"id": "T16", "level": "L2", "question": "备件 SP-001 都用在哪些设备上？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/spare/bom/by-spare", "params": {"spareId": "SP-001"}}],
         "G2": [{"tool": "get_equipment_spare_bom", "arguments": {"spareId": "SP-001"}}],
         "G3": "QUERY equipment_spare_bom WHERE spare_id = 'SP-001' JOIN equipment"
     }},
    {"id": "T17", "level": "L2", "question": "设备 EQ-001 最近一次保养是什么时候？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/maintenance/task/page", "params": {"equipmentId": "EQ-001", "status": "completed"}}],
         "G2": [{"tool": "query_maintenance_tasks", "arguments": {"equipmentId": "EQ-001", "status": 2}}],
         "G3": "QUERY maintenance_task WHERE equipment_id = 'EQ-001' AND status = 'completed' ORDER BY actual_time DESC LIMIT 1"
     }},
    {"id": "T18", "level": "L2", "question": "本月新增了多少故障报修？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/fault-report/page", "params": {"reportTimeBegin": "2026-03-01"}}],
         "G2": [{"tool": "query_fault_reports", "arguments": {"dateRange": {"start": "2026-03-01", "end": "2026-03-25"}}}],
         "G3": "QUERY fault_report WHERE report_time >= '2026-03-01' AGGREGATE COUNT"
     }},
    {"id": "T19", "level": "L2", "question": "设备 EQ-003 的全生命周期事件有哪些？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/equipment/lifecycle-timeline", "params": {"equipmentId": "EQ-003"}}],
         "G2": [{"tool": "get_equipment_lifecycle", "arguments": {"equipmentId": "EQ-003"}}],
         "G3": "UNSUPPORTED"
     }},
    {"id": "T20", "level": "L2", "question": "近 30 天巡检发现最多异常的设备是哪台？",
     "ground_truth": {
         "G1": [{"method": "GET /eam/patrol/task/analytics/by-equipment", "params": {"days": 30}}],
         "G2": [{"tool": "get_patrol_analytics", "arguments": {"days": 30, "dimension": "by_equipment"}}],
         "G3": "QUERY anomaly_record WHERE source = 'patrol' AND create_time >= '2026-02-23' AGGREGATE COUNT ORDER BY COUNT DESC LIMIT 1"
     }},

    # === L3 链式多跳 ===
    {"id": "T21", "level": "L3", "question": "A 线上月维修用了哪些备件？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/equipment/page", "params": {"productionLineId": "A线"}},
             {"method": "GET /eam/repair-order/page", "params": {"equipmentId": "...", "dateRange": "2026-02"}},
             {"method": "GET /eam/repair-order/spare/list", "params": {"repairOrderId": "..."}}
         ],
         "G2": [
             {"tool": "query_equipment", "arguments": {"productionLineId": "A线"}},
             {"tool": "query_repair_orders", "arguments": {"equipmentId": "...", "dateRange": {"start": "2026-02-01", "end": "2026-02-28"}}},
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "..."}}
         ],
         "G3": "QUERY repair_spare_usage JOIN repair_order JOIN equipment JOIN production_line_equipment WHERE production_line_equipment.production_line_id = 'A线' AND repair_order.create_time BETWEEN '2026-02-01' AND '2026-02-28'"
     }},
    {"id": "T22", "level": "L3", "question": "设备 EQ-003 的 BOM 里哪些备件库存不足？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/spare/bom/by-equipment", "params": {"equipmentId": "EQ-003"}},
             {"method": "GET /eam/spare/part/stock-summary", "params": {"ids": "..."}}
         ],
         "G2": [
             {"tool": "get_equipment_spare_bom", "arguments": {"equipmentId": "EQ-003"}},
             {"tool": "get_spare_stock", "arguments": {"spareId": "..."}}
         ],
         "G3": "QUERY equipment_spare_bom JOIN spare_part JOIN spare_stock WHERE equipment_spare_bom.equipment_id = 'EQ-003' AND spare_stock.quantity < spare_part.safety_stock"
     }},
    {"id": "T23", "level": "L3", "question": "上月维修时参考了哪些知识文档？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/repair-order/page", "params": {"dateRange": "2026-02"}},
             {"method": "GET /eam/repair-order/knowledge-ref/list", "params": {"repairOrderId": "..."}}
         ],
         "G2": [
             {"tool": "query_repair_orders", "arguments": {"dateRange": {"start": "2026-02-01", "end": "2026-02-28"}}},
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "..."}}
         ],
         "G3": "QUERY repair_order JOIN repair_order_knowledge_ref WHERE repair_order.create_time BETWEEN '2026-02-01' AND '2026-02-28'"
     }},
    {"id": "T24", "level": "L3", "question": "B 线上个月巡检发现了几次异常？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/equipment/page", "params": {"productionLineId": "B线"}},
             {"method": "GET /eam/anomaly/grouped", "params": {"source": "patrol"}}
         ],
         "G2": [
             {"tool": "query_equipment", "arguments": {"productionLineId": "B线"}},
             {"tool": "query_anomaly_records", "arguments": {"source": "patrol"}}
         ],
         "G3": "QUERY anomaly_record JOIN equipment JOIN production_line_equipment WHERE production_line_equipment.production_line_id = 'B线' AND anomaly_record.source = 'patrol' AND anomaly_record.create_time BETWEEN '2026-02-01' AND '2026-02-28' AGGREGATE COUNT"
     }},
    {"id": "T25", "level": "L3", "question": "最近维修用量最大的备件，关联了哪些设备？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/repair-order/page", "params": {}},
             {"method": "GET /eam/repair-order/spare/list", "params": {"repairOrderId": "..."}},
             {"method": "GET /eam/spare/bom/by-spare", "params": {"spareId": "..."}}
         ],
         "G2": [
             {"tool": "query_repair_orders", "arguments": {}},
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "..."}},
             {"tool": "get_equipment_spare_bom", "arguments": {"spareId": "..."}}
         ],
         "G3": "QUERY repair_spare_usage JOIN spare_part JOIN equipment_spare_bom JOIN equipment AGGREGATE SUM(quantity) ORDER BY SUM DESC LIMIT 1"
     }},
    {"id": "T26", "level": "L3", "question": "A 线设备的保养计划执行率是多少？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/equipment/page", "params": {"productionLineId": "A线"}},
             {"method": "GET /eam/maintenance/task/page", "params": {"equipmentId": "..."}}
         ],
         "G2": [
             {"tool": "query_equipment", "arguments": {"productionLineId": "A线"}},
             {"tool": "query_maintenance_tasks", "arguments": {"equipmentId": "..."}}
         ],
         "G3": "QUERY maintenance_task JOIN equipment JOIN production_line_equipment WHERE production_line_equipment.production_line_id = 'A线' AGGREGATE COUNT"
     }},
    {"id": "T27", "level": "L3", "question": "维修工单 WO-005 的出库单涉及了哪些仓库？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/repair-order/related-stock-out/list", "params": {"repairOrderId": "WO-005"}},
             {"method": "GET /eam/repair-order/related-stock-out/items", "params": {"stockOutOrderId": "..."}}
         ],
         "G2": [
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "WO-005"}},
             {"tool": "query_spare_transactions", "arguments": {"type": "stock_out"}}
         ],
         "G3": "QUERY stock_out_order JOIN repair_order WHERE repair_order.id = 'WO-005'"
     }},
    {"id": "T28", "level": "L3", "question": "设备 EQ-001 最近一次故障的维修花了多少工时？",
     "ground_truth": {
         "G1": [
             {"method": "GET /eam/fault-report/page", "params": {"equipmentId": "EQ-001"}},
             {"method": "GET /eam/repair-order/get-by-fault-report-id", "params": {"faultReportId": "..."}},
             {"method": "GET /eam/repair-order/workload/list", "params": {"repairOrderId": "..."}}
         ],
         "G2": [
             {"tool": "query_fault_reports", "arguments": {"equipmentId": "EQ-001"}},
             {"tool": "get_repair_detail", "arguments": {"faultReportId": "..."}}
         ],
         "G3": "QUERY repair_workload JOIN repair_order JOIN fault_report WHERE fault_report.equipment_id = 'EQ-001' ORDER BY fault_report.report_time DESC LIMIT 1"
     }},

    # === L4 跨域关联 ===
    {"id": "T29", "level": "L4", "question": "故障率最高的设备，保养是否按计划执行？",
     "ground_truth": {
         "G2": [
             {"tool": "query_fault_reports", "arguments": {}},
             {"tool": "query_maintenance_tasks", "arguments": {"equipmentId": "..."}}
         ]
     }},
    {"id": "T30", "level": "L4", "question": "上月维修成本最高的设备，它的巡检有没有发现过异常？",
     "ground_truth": {
         "G2": [
             {"tool": "query_repair_orders", "arguments": {"dateRange": {"start": "2026-02-01", "end": "2026-02-28"}}},
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "..."}},
             {"tool": "query_anomaly_records", "arguments": {"source": "patrol"}}
         ]
     }},
    {"id": "T31", "level": "L4", "question": "备件库存预警涉及的设备中，有哪些正在维修？",
     "ground_truth": {
         "G2": [
             {"tool": "get_spare_alerts", "arguments": {}},
             {"tool": "get_equipment_spare_bom", "arguments": {"spareId": "..."}},
             {"tool": "query_equipment", "arguments": {"status": 2}}
         ]
     }},
    {"id": "T32", "level": "L4", "question": "A 线设备的故障、保养、巡检三项指标概览",
     "ground_truth": {
         "G2": [
             {"tool": "query_equipment", "arguments": {"productionLineId": "A线"}},
             {"tool": "query_fault_reports", "arguments": {}},
             {"tool": "query_maintenance_tasks", "arguments": {}},
             {"tool": "get_patrol_analytics", "arguments": {"dimension": "by_equipment"}}
         ]
     }},
    {"id": "T33", "level": "L4", "question": "近 3 个月有故障但没安排保养的设备有哪些？",
     "ground_truth": {
         "G2": [
             {"tool": "query_fault_reports", "arguments": {"dateRange": {"start": "2026-01-01"}}},
             {"tool": "query_maintenance_tasks", "arguments": {"dateRange": {"start": "2026-01-01"}}}
         ]
     }},
    {"id": "T34", "level": "L4", "question": "外协维修的设备中，有没有重点设备？",
     "ground_truth": {
         "G2": [
             {"tool": "query_repair_orders", "arguments": {}},
             {"tool": "get_equipment_detail", "arguments": {"equipmentId": "..."}}
         ]
     }},

    # === L5 聚合+时间推理 ===
    {"id": "T35", "level": "L5", "question": "哪条产线近 3 个月故障呈上升趋势，且备件库存不足？",
     "ground_truth": {
         "G2": [
             {"tool": "get_fault_trend", "arguments": {"days": 90}},
             {"tool": "query_equipment", "arguments": {}},
             {"tool": "get_spare_alerts", "arguments": {}}
         ]
     }},
    {"id": "T36", "level": "L5", "question": "上季度各产线的保养完成率排名？",
     "ground_truth": {
         "G2": [
             {"tool": "query_equipment", "arguments": {}},
             {"tool": "query_maintenance_tasks", "arguments": {"dateRange": {"start": "2025-10-01", "end": "2025-12-31"}}}
         ]
     }},
    {"id": "T37", "level": "L5", "question": "同比去年同期，今年 Q1 的故障报修数是增还是减？",
     "ground_truth": {
         "G2": [
             {"tool": "query_fault_reports", "arguments": {"dateRange": {"start": "2026-01-01", "end": "2026-03-31"}}},
             {"tool": "query_fault_reports", "arguments": {"dateRange": {"start": "2025-01-01", "end": "2025-03-31"}}}
         ]
     }},
    {"id": "T38", "level": "L5", "question": "近半年维修频次最高的 3 台设备，各自的平均维修周期是多少？",
     "ground_truth": {
         "G2": [
             {"tool": "query_repair_orders", "arguments": {"dateRange": {"start": "2025-09-25", "end": "2026-03-25"}}},
             {"tool": "get_equipment_detail", "arguments": {"equipmentId": "..."}}
         ]
     }},
    {"id": "T39", "level": "L5", "question": "备件月消耗量环比分析，哪些备件用量在持续上升？",
     "ground_truth": {
         "G2": [
             {"tool": "query_repair_orders", "arguments": {"dateRange": {"start": "2025-10-01"}}},
             {"tool": "get_repair_detail", "arguments": {"repairOrderId": "..."}}
         ]
     }},
    {"id": "T40", "level": "L5", "question": "近 6 个月巡检异常率变化趋势，有没有季节性规律？",
     "ground_truth": {
         "G2": [
             {"tool": "get_patrol_analytics", "arguments": {"days": 180, "dimension": "overview"}},
             {"tool": "query_anomaly_records", "arguments": {"source": "patrol"}}
         ]
     }}
]

print(f"测试集加载完成: {len(test_cases)} 题")
for level in ['L1', 'L2', 'L3', 'L4', 'L5']:
    count = len([t for t in test_cases if t['level'] == level])
    print(f"  {level}: {count} 题")

## 2. Prompt 模板

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """你是 EAM（设备资产管理）系统的 AI 助手。用户会用自然语言提出查询或分析需求。

你的任务是：
1. 理解用户意图
2. 从下方工具清单中选择需要调用的工具
3. 填入正确的参数

规则：
- 只能使用下方清单中列出的工具，不能编造不存在的工具
- 如果需要多个工具配合才能回答，按执行顺序列出所有需要的工具调用
- 如果没有合适的工具能回答用户问题，返回 {{"unsupported": true, "reason": "..."}}
- 只返回 JSON，不要解释

{tool_catalog}
"""

In [ ]:
# G2 MCP Tool 清单 (核心实验)
G2_TOOL_CATALOG = """
输出格式：
```json
{"calls": [{"tool": "tool_name", "arguments": {"参数名": "值"}}]}
```

可用工具：

1. query_equipment — 按条件查询设备列表
   参数: keyword(string), status(int: 0=待验收 1=运行中 2=维修中 3=停机 4=封存 5=待整改 6=闲置 7=报废), categoryId(int), locationId(int), productionLineId(int), deptId(int), isKey(int: 0=否 1=是)

2. get_equipment_detail — 获取单台设备完整详情+KPI（故障次数、维修时长、保养执行率等）
   参数: equipmentId(int, 必填)

3. get_equipment_lifecycle — 获取设备全生命周期时间线（购置、验收、状态变更、故障、维修、保养等）
   参数: equipmentId(int, 必填)

4. get_equipment_status_distribution — 获取设备状态分布统计（各状态设备数量）
   参数: 无

5. query_fault_reports — 查询故障报修列表
   参数: equipmentId(int), status(int: 0=待审核 1=已通过 2=已拒绝 3=已撤回), faultTypeId(int), dateRange(object: {start, end})

6. query_repair_orders — 查询维修工单列表
   参数: equipmentId(int), status(int: 0=待分配 1=待接单 2=待维修 3=维修中 4=待验收 5=已验收 6=已关闭), dateRange(object: {start, end})

7. get_repair_detail — 获取维修工单完整详情（基本信息+备件清单+工时记录+知识引用）
   参数: repairOrderId(int) 或 faultReportId(int)，二选一

8. get_fault_trend — 故障趋势分析（按日统计报修数量）
   参数: days(int, 默认30), equipmentId(int, 可选), productionLineId(int, 可选)

9. query_maintenance_tasks — 查询保养任务列表
   参数: equipmentId(int), status(int: 0=待执行 1=执行中 2=已完成 3=已跳过), planId(int), dateRange(object: {start, end})

10. get_maintenance_detail — 获取保养任务详情+执行记录
    参数: taskId(int, 必填)

11. query_patrol_tasks — 查询巡检任务列表
    参数: equipmentId(int), status(int), planId(int), dateRange(object: {start, end})

12. get_patrol_analytics — 巡检综合分析
    参数: days(int, 默认30), dimension(string: overview|by_plan|by_equipment|by_standard, 默认overview)

13. query_anomaly_records — 查询异常记录
    参数: keyword(string), source(string: 巡检/保养/其他), severity(int), status(int), grouped(bool, 默认false)

14. get_anomaly_statistics — 异常统计KPI（总数、待处理、处理率、严重级别分布）
    参数: 无

15. query_spare_parts — 查询备件列表
    参数: keyword(string), typeId(int), status(int)

16. get_spare_stock — 查询备件库存（按备件ID或仓库ID查）
    参数: spareId(int), warehouseId(int)

17. get_equipment_spare_bom — 查询设备与备件BOM关联（正向查设备的备件，反向查备件的设备）
    参数: equipmentId(int), spareId(int)

18. query_spare_transactions — 查询备件流转记录
    参数: type(string: stock_in|stock_out|transfer|return|scrap|purchase), spareId(int), dateRange(object: {start, end})

19. get_spare_alerts — 查询库存预警（库存低于安全线的备件）
    参数: 无

20. get_dashboard_summary — 系统总览（设备总数、运行中/故障/报废数、待处理报修/工单/保养数）
    参数: 无

21. get_governance_dashboard — 设备生命周期治理看板
    参数: startTime(datetime), endTime(datetime), keyword(string), drilldown(bool, 默认false)

22. get_todo_list — 待办事项（待审核报修+待接单工单+待执行保养）
    参数: 无
"""

## 3. 实验执行引擎

In [ ]:
import time

def run_experiment(group: str, tool_catalog: str, model: str = "gpt-4o-mini"):
    """对一组实验跑全部测试集"""
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(tool_catalog=tool_catalog)
    results = []
    
    for i, tc in enumerate(test_cases):
        print(f"  [{i+1}/{len(test_cases)}] {tc['id']}: {tc['question'][:30]}...", end=" ")
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": tc["question"]}
                ],
                temperature=0,
                max_tokens=1000
            )
            output = resp.choices[0].message.content.strip()
            # 尝试提取 JSON
            if "```json" in output:
                output = output.split("```json")[1].split("```")[0].strip()
            elif "```" in output:
                output = output.split("```")[1].split("```")[0].strip()
            
            try:
                parsed = json.loads(output)
            except json.JSONDecodeError:
                parsed = {"raw": output, "parse_error": True}
            
            results.append({
                "id": tc["id"],
                "level": tc["level"],
                "question": tc["question"],
                "output": parsed,
                "raw": output,
                "ground_truth": tc["ground_truth"].get(group)
            })
            print("✓")
        except Exception as e:
            print(f"✗ {e}")
            results.append({
                "id": tc["id"],
                "level": tc["level"],
                "question": tc["question"],
                "output": None,
                "error": str(e),
                "ground_truth": tc["ground_truth"].get(group)
            })
        
        time.sleep(0.5)  # 避免限流
    
    return results

## 4. 运行 G2 实验（核心）

In [ ]:
print("=== G2 MCP Tool 路由实验 ===")
print(f"模型: gpt-4o-mini | 测试集: {len(test_cases)} 题")
print()
g2_results = run_experiment("G2", G2_TOOL_CATALOG)

In [ ]:
# 查看 G2 原始输出
for r in g2_results:
    print(f"\n{'='*60}")
    print(f"{r['id']} [{r['level']}] {r['question']}")
    print(f"LLM 输出: {json.dumps(r['output'], ensure_ascii=False, indent=2)}")
    if r.get('ground_truth'):
        print(f"期望:     {json.dumps(r['ground_truth'], ensure_ascii=False)}")

## 5. 评分

In [ ]:
def score_g2_result(result):
    """对 G2 单题评分，返回四维度分数"""
    output = result.get("output")
    gt = result.get("ground_truth")
    
    if not output or not gt or output.get("parse_error"):
        return {"d1_recall": 0, "d2_precision": 0, "d3_params": 0, "d4_filters": 0, "notes": "解析失败"}
    
    # 提取 LLM 选的 tools
    calls = output.get("calls", [])
    if isinstance(output, dict) and "tool" in output:
        calls = [output]  # 单个调用
    
    llm_tools = set()
    for c in calls:
        tool_name = c.get("tool", "")
        if tool_name:
            llm_tools.add(tool_name)
    
    # 提取期望的 tools
    gt_tools = set()
    for g in gt:
        gt_tools.add(g.get("tool", ""))
    
    # D1: 召回率 — 期望的 tools 中，LLM 选到了几个
    if gt_tools:
        d1 = len(llm_tools & gt_tools) / len(gt_tools)
    else:
        d1 = 1.0
    
    # D2: 精确率 — LLM 选的 tools 中，有多少是正确的
    if llm_tools:
        d2 = len(llm_tools & gt_tools) / len(llm_tools)
    else:
        d2 = 0.0
    
    # D3/D4: 参数和过滤条件 — 需要人工评分，先标记为待评
    level = result.get("level", "L1")
    d3 = None if level in ["L2", "L3", "L4", "L5"] else 1.0  # L1 不评参数
    d4 = None if level in ["L3", "L4", "L5"] else 1.0  # L1/L2 不评过滤条件
    
    return {
        "d1_recall": round(d1, 2),
        "d2_precision": round(d2, 2),
        "d3_params": d3,
        "d4_filters": d4,
        "llm_tools": list(llm_tools),
        "gt_tools": list(gt_tools),
        "notes": "D3/D4 待人工评分" if d3 is None or d4 is None else ""
    }


# 自动评分 D1/D2
g2_scores = []
for r in g2_results:
    score = score_g2_result(r)
    score["id"] = r["id"]
    score["level"] = r["level"]
    g2_scores.append(score)

# 按级别汇总
print("\n=== G2 自动评分汇总（D1 召回 / D2 精确） ===")
print(f"{'级别':>4} | {'D1 召回':>8} | {'D2 精确':>8} | {'题数':>4}")
print("-" * 35)
for level in ['L1', 'L2', 'L3', 'L4', 'L5']:
    level_scores = [s for s in g2_scores if s['level'] == level]
    if level_scores:
        avg_d1 = sum(s['d1_recall'] for s in level_scores) / len(level_scores)
        avg_d2 = sum(s['d2_precision'] for s in level_scores) / len(level_scores)
        print(f"{level:>4} | {avg_d1:>7.1%} | {avg_d2:>7.1%} | {len(level_scores):>4}")

all_d1 = sum(s['d1_recall'] for s in g2_scores) / len(g2_scores)
all_d2 = sum(s['d2_precision'] for s in g2_scores) / len(g2_scores)
print("-" * 35)
print(f"{'总计':>4} | {all_d1:>7.1%} | {all_d2:>7.1%} | {len(g2_scores):>4}")

In [ ]:
# 查看每题得分明细
for s in g2_scores:
    status = "✓" if s['d1_recall'] >= 0.8 and s['d2_precision'] >= 0.8 else "△" if s['d1_recall'] >= 0.5 else "✗"
    print(f"{status} {s['id']} [{s['level']}] D1={s['d1_recall']:.0%} D2={s['d2_precision']:.0%} | LLM: {s['llm_tools']} | 期望: {s['gt_tools']}")

## 6. 保存结果

In [ ]:
# 保存原始结果和评分
import datetime

output_data = {
    "experiment": "G2_MCP_Tool",
    "model": "gpt-4o-mini",
    "timestamp": datetime.datetime.now().isoformat(),
    "results": g2_results,
    "scores": g2_scores,
    "summary": {
        "total": len(g2_scores),
        "avg_d1_recall": round(all_d1, 3),
        "avg_d2_precision": round(all_d2, 3)
    }
}

output_path = "g2_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"结果已保存到 {output_path}")